# 01 – Data exploration: ICA Historical Catastrophe List

This notebook explores the raw ICA catastrophe list before cleaning. It checks:

1. how many events have a recorded loss,
2. how complete the location (State and Postcode) fields are, and
3. how large ICA's normalisation factors are compared with inflation alone.

The findings shape the cleaning steps in `src/` and the analysis in `02_analysis.ipynb`.

In [1]:
# Imports and settings
from pathlib import Path

import pandas as pd

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 30

# Notebook runs from /notebooks, so go up one level to the project root
PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / "data" / "raw"
ICA_FILE = RAW / "ICA-Historical-Normalised-Catastrophe-2026_09.xlsx"

In [2]:
# Load the detailed catastrophe sheet (column headers are on Excel row 11)
ica = pd.read_excel(ICA_FILE, sheet_name="ICA_DETAILED_HISTORICAL_CATASTR", header=10)
ica.shape

(747, 23)

In [3]:
# Column overview: non-null counts and data types
ica.info()

<class 'pandas.DataFrame'>
RangeIndex: 747 entries, 0 to 746
Data columns (total 23 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   CAT Name                      747 non-null    str           
 1   Event Name                    747 non-null    str           
 2   Event Start                   747 non-null    datetime64[us]
 3   Event Finish                  745 non-null    object        
 4   Financial Year                747 non-null    str           
 5   State                         233 non-null    str           
 6   Town                          61 non-null     str           
 7   Description                   747 non-null    str           
 8   Type                          747 non-null    str           
 9   Year                          747 non-null    int64         
 10  Postcode                      65 non-null     str           
 11  ORIGINAL LOSS VALUE           349 non-null 

In [4]:
# Number of events by peril
ica["Type"].value_counts()

Type
Cyclone       458
Storm         116
Flood          66
Bushfire       51
Hail           36
Earthquake     16
Man-made        4
Name: count, dtype: int64

In [5]:
# True if the event has a loss value recorded
has_loss = ica["ORIGINAL LOSS VALUE"].notna()
print(f"Events with a loss value: {has_loss.sum()} of {len(ica)}")

Events with a loss value: 349 of 747


In [6]:
# Event category = first 3 letters of CAT Name: "CAT", "SE2" or "Und" (Undeclared)
pd.crosstab(ica["CAT Name"].str[:3], has_loss)

ORIGINAL LOSS VALUE,False,True
CAT Name,,
CAT,3,234
SE2,0,13
Und,395,102


In [7]:
# Raw values in the State column, including blanks
ica["State"].value_counts(dropna=False)

State
NaN                  514
FNQ                  167
SEQ                   40
NSW                    5
SWQ                    4
QLD                    3
VIC                    2
NSW and SEQ            2
WA                     2
NT                     1
Outside Australia      1
QLD, NSW               1
NSW, QLD               1
SA                     1
VIC, NSW, TAS          1
SA, Vic                1
Vic                    1
Name: count, dtype: int64

In [8]:
# Location coverage among events with a recorded loss
losses = ica[has_loss]
pd.crosstab(
    losses["State"].isna(),
    losses["Postcode"].isna(),
    rownames=["State missing"],
    colnames=["Postcode missing"],
)

Postcode missing,False,True
State missing,,
False,34,66
True,28,221


In [9]:
# Normalisation factor by decade
losses = losses.assign(
    norm_factor=losses["NORMALISED LOSS VALUE (2022)"] / losses["ORIGINAL LOSS VALUE"]
)
losses.groupby(losses["Year"] // 10 * 10)["norm_factor"].median()

Year
1960   101.12
1970    45.36
1980    13.44
1990     6.44
2000     2.56
2010     1.56
2020     1.00
Name: norm_factor, dtype: float64

### Insight: inflation explains only part of the normalisation factor

For a typical 1974 event, ICA's normalisation factor is about **45×**, but consumer
prices (CPI) rose only about **8×** between 1974 and 2022. The remaining **~5.6×**
(45 ÷ 8) reflects **exposure growth**: there is more property at risk today, and each
property is worth more than general price growth alone would suggest.

Likely drivers:
- **More people and dwellings**: Australia's population roughly doubled
  (about 13.7 million in 1974 to 26 million in 2022).
- **Larger, more valuable homes**: house prices have grown much faster than CPI.
- **More insured contents per home**: electronics, furniture and vehicles.
- **Higher insurance take-up and development in exposed areas**, such as coastal
  and bushland fringes.

**Implication:** trends in *original* losses mostly reflect exposure growth.
To test whether hazards themselves are getting worse, the analysis uses
*normalised* losses.